In [19]:
import pandas as pd 
df =  pd.read_csv("../DATA/clean_data.csv")

In [21]:
df.shape

(29342, 12)

In [23]:
df

,url,year,type_boit,type_carburant,kilometrage,marke,model,puissance,premiere_main,Nombre_doors,city,price
0,https://www.avito.ma/fr/ain_sebaa/voitures_d'o...,2021,automatique,essence,62499,fiat,500,7,2,5.0,casablanca,159000
1,https://www.avito.ma/fr/al_maghrib_al_arabi/vo...,2018,manuelle,diesel,154999,hyundai,creta,6,2,5.0,kénitra,175000
2,https://www.avito.ma/fr/ain_sebaa/voitures_d'o...,2021,automatique,diesel,97499,dacia,duster,6,2,5.0,casablanca,187000
3,https://www.avito.ma/fr/ain_sebaa/voitures_d'o...,2022,manuelle,diesel,62499,peugeot,208,6,2,5.0,casablanca,168000
4,https://www.avito.ma/fr/ain_sebaa/voitures_d'o...,2022,manuelle,diesel,72499,renault,clio,6,2,5.0,casablanca,162000
...,...,...,...,...,...,...,...,...,...,...,...,...
29337,https://www.avito.ma/fr/sidi_bernoussi/voiture...,2014,manuelle,diesel,164999,peugeot,301,8,0,5.0,casablanca,90000
29338,https://www.avito.ma/fr/sidi_othmane/voitures_...,2017,automatique,diesel,154999,kia,sportage,8,0,5.0,casablanca,169998
29339,https://www.avito.ma/fr/sidi_othmane/voitures_...,2017,automatique,diesel,154999,kia,sportage,8,0,5.0,casablanca,170000
29340,https://www.avito.ma/fr/boufkrane/voitures_d'o...,2007,manuelle,diesel,32499,mitsubishi,l200,10,0,5.0,boufkrane,70000


In [25]:
df.isnull().sum()

url               0
year              0
type_boit         0
type_carburant    0
kilometrage       0
marke             0
model             0
puissance         0
premiere_main     0
Nombre_doors      0
city              0
price             0
dtype: int64

In [159]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import joblib
import warnings
from IPython.display import display, HTML

In [161]:
!pip install xgboost

In [163]:
# =============================================
data = df  # Make sure df is defined earlier

X = data.drop('price', axis=1)
y = data['price']

# Split into train (60%), val (20%), test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)


In [165]:
 # Preprocessing
# =============================================
numeric_features = ['year', 'kilometrage', 'puissance', 'Nombre_doors']
categorical_features = ['type_boit', 'type_carburant', 'marke', 'model', 'premiere_main']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
], remainder='drop')

# ================

In [167]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(
        objective='reg:squarederror',
        tree_method='hist',
        eval_metric='rmse',
        random_state=42
    ))
])

param_dist = {
    'regressor__max_depth': [3, 5],
    'regressor__learning_rate': [0.05, 0.1],
    'regressor__subsample': [0.8, 1.0],
    'regressor__colsample_bytree': [0.8, 1.0],
    'regressor__gamma': [0, 0.1]
}



In [169]:
search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='neg_root_mean_squared_error',
    verbose=2,
    random_state=42,
    n_jobs=-1
)

In [171]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    search.fit(X_train, y_train)

# Extract best parameters
best_params = search.best_params_


Fitting 3 folds for each of 10 candidates, totalling 30 fits


In [173]:
preprocessor.fit(X_train)
X_train_preprocessed = preprocessor.transform(X_train)
X_val_preprocessed = preprocessor.transform(X_val)

# Clean param keys
clean_params = {k.replace('regressor__', ''): v for k, v in best_params.items()}

final_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    tree_method='hist',
    eval_metric='rmse',
    early_stopping_rounds=10,
    random_state=42,
    **clean_params
)

In [175]:
final_model.fit(X_train_preprocessed, y_train, eval_set=[(X_val_preprocessed, y_val)], verbose=False)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=10,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=0.1, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [177]:
def generate_performance_report(model, X_true, y_true, set_name):
    y_pred = model.predict(X_true)

    metrics = {
        'MSE': mean_squared_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R²': r2_score(y_true, y_pred)
    }

    # Visualization
    plt.figure(figsize=(12, 5))

    # Actual vs Predicted
    plt.subplot(1, 2, 1)
    sns.regplot(x=y_true, y=y_pred, scatter_kws={'alpha': 0.3})
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--')
    plt.xlabel('Actual Prices')
    plt.ylabel('Predicted Prices')
    plt.title(f'{set_name} Set: Actual vs Predicted')

    # Error Distribution
    plt.subplot(1, 2, 2)
    residuals = y_true - y_pred
    sns.histplot(residuals, kde=True, bins=30)
    plt.xlabel('Prediction Errors')
    plt.title('Error Distribution')

    plt.suptitle(f'{set_name} Set Performance', y=1.02)
    plt.tight_layout()

    return pd.DataFrame([metrics], index=[set_name])


In [183]:
# After your existing code, add this model saving block

try:
    import joblib
        # Save both the model and feature names
    joblib.dump(best_model, 'Xgboost.pkl')

    print("\nModel successfully saved!")
except Exception as e:
    print(f"\nModel saving failed: {str(e)}")


Model successfully saved!


In [185]:
import pandas as pd
import joblib
from datetime import datetime

# Helper function to clean numeric values
def clean_numeric(value):
    if isinstance(value, str):
        return float(''.join(filter(str.isdigit, value)) or 0)
    return float(value)

# Helper function to categorize power
def categorize_power(p):
    if p <= 6:
        return 'Small'
    elif 7 <= p <= 8:
        return 'Medium'
    return 'Large'

def prepare_user_input(user_data):
    # Make a copy to avoid modifying original input
    data = user_data.copy()
    current_year = datetime.now().year

    # Clean and transform specific fields
    if '-' in str(data['kilometrage']):
        data['kilometrage'] = sum(
            float(x.replace(" ", "").strip()) for x in str(data['kilometrage']).split('-')
        ) / 2
    else:
        data['kilometrage'] = clean_numeric(data['kilometrage'])

    data['puissance'] = clean_numeric(data['puissance'])
    data['Age voiture'] = current_year - clean_numeric(data['year'])

    puissance_cat = categorize_power(data['puissance'])
    is_luxury = 1 if data.get('marke', '').lower() in ['bmw', 'audi', 'mercedes'] else 0

    # Construct the DataFrame
    final_input = pd.DataFrame([{
        'year': clean_numeric(data['year']),
        'type_boit': data['type_boit'].title(),
        'type_carburant': data['type_carburant'].title(),
        'kilometrage': data['kilometrage'],
        'marke': data['marke'].title(),
        'model': data['model'].title(),
        'puissance': data['puissance'],
        'premiere_main': data['premiere_main'].title(),
        'Nombre_doors': clean_numeric(data['Nombre_doors']),
    }])

    return final_input

# Example usage
user_input = {
    'year': 2017,
    'type_boit': 'manuelle',
    'type_carburant': 'diesel',
    'kilometrage': '100 000 - 120 999',
    'marke': 'dacia',
    'model': 'logan',
    'puissance': '6 cv',
    'premiere_main': 'non',
    'Nombre_doors': '5',
    'city': 'casablanca'
}

# Load model pipeline
model = joblib.load('xgboost.pkl')

# Prepare data and predict
prepared_data = prepare_user_input(user_input)
prediction = model.predict(prepared_data)

print(f"✅ Predicted Price: {prediction[0]:,.2f} MAD")


✅ Predicted Price: 109,578.84 MAD
